<a href="https://colab.research.google.com/github/oxedanda/pml_final_project/blob/main/notebooks/04_future_forecast_simulator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Experimental Future Wine Production Simulator
## Forecasting the 2026/27 campaign by Portuguese viticultural region

This notebook makes a genuine one-year-ahead forecast. It uses only information available before the target campaign: regional production lags, the recent three-year mean, the latest production trend, vineyard area, region, and year.

Model selection uses rolling-origin validation over 2018-2022. Campaigns 2023-2025 are retained for a final one-step-ahead test. The simulator is experimental: its uncertainty interval is based on historical validation errors and is not a guarantee.

In [ ]:
# prompt: Set up this Colab notebook reproducibly without deleting the existing notebook 03.
from pathlib import Path

REPO_DIR = Path("/content/pml_final_project")
if REPO_DIR.exists():
    %cd /content/pml_final_project
    !git pull --ff-only
else:
    !git clone https://github.com/oxedanda/pml_final_project.git /content/pml_final_project
    %cd /content/pml_final_project

!pip -q install -r requirements.txt

## 1. Build lagged features and evaluate candidate models

For each target campaign, `production_lag_1`, `production_lag_2`, and the rolling mean are calculated from earlier campaigns only. The evaluation therefore does not reveal the target value to the model.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from src.future_forecast import (
    evaluate_forecasters,
    forecast_2026,
    load_history,
)

evaluation = evaluate_forecasters()
print("Selected on rolling validation:", evaluation.selected_model)
print(f"Empirical 90% interval half-width: {evaluation.interval_half_width:,.0f} hl")

In [ ]:
validation_metrics = (
    evaluation.metrics
    .query("split == 'rolling_validation_2018_2022'")
    .sort_values("MAE")
)
test_metrics = (
    evaluation.metrics
    .query("split == 'test_2023_2025'")
    .sort_values("MAE")
)

print("Rolling validation: 2018-2022")
display(validation_metrics[["model", "MAE", "RMSE", "R2"]])
print("Final one-step-ahead test: 2023-2025")
display(test_metrics[["model", "MAE", "RMSE", "R2"]])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for axis, table, title in [
    (axes[0], validation_metrics, "Rolling validation MAE"),
    (axes[1], test_metrics, "Final test MAE"),
]:
    axis.barh(table["model"], table["MAE"], color="#7a0019")
    axis.invert_yaxis()
    axis.set_xlabel("MAE (hectolitres)")
    axis.set_title(title)
    axis.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

## 2. Forecast all regions for 2026/27

The default scenario assumes that 2026 vineyard area remains equal to the latest available area for each region. This assumption can be changed in the interactive simulator below.

In [ ]:
future_forecasts = forecast_2026(evaluation)
display(future_forecasts.style.format({
    "vineyard_area_ha": "{:,.0f}",
    "predicted_production_hl": "{:,.0f}",
    "lower_90_hl": "{:,.0f}",
    "upper_90_hl": "{:,.0f}",
}))

## 3. Interactive regional scenario

Choose a region and optionally change its expected vineyard area. The model is refitted on all supervised observations through 2025 before producing the 2026/27 estimate.

In [ ]:
# prompt: Add a small transparent interface for region and vineyard-area scenarios.
import ipywidgets as widgets

history = load_history()
latest_area = (
    history.dropna(subset=["vineyard_area_ha"])
    .sort_values("year_start")
    .groupby("region", observed=True)
    .tail(1)
    .set_index("region")["vineyard_area_ha"]
)

region_widget = widgets.Dropdown(
    options=sorted(latest_area.index),
    description="Region:",
)
area_widget = widgets.FloatText(
    value=float(latest_area.loc[region_widget.value]),
    description="Area (ha):",
)
forecast_button = widgets.Button(description="Forecast 2026/27", button_style="primary")
forecast_output = widgets.Output()

def update_default_area(change):
    area_widget.value = float(latest_area.loc[change["new"]])

def show_forecast(_button):
    with forecast_output:
        forecast_output.clear_output()
        result = forecast_2026(
            evaluation,
            area_overrides={region_widget.value: area_widget.value},
        )
        row = result[result["region"] == region_widget.value]
        display(row.style.format({
            "vineyard_area_ha": "{:,.0f}",
            "predicted_production_hl": "{:,.0f}",
            "lower_90_hl": "{:,.0f}",
            "upper_90_hl": "{:,.0f}",
        }))

region_widget.observe(update_default_area, names="value")
forecast_button.on_click(show_forecast)
display(widgets.VBox([region_widget, area_widget, forecast_button, forecast_output]))

## Interpretation and limitations

- This is a one-year-ahead experimental forecast, not a guaranteed production value.
- The model uses actual production from the preceding campaign, so it should be rerun when newer official IVV data become available.
- The 90% interval is an empirical error band derived from rolling-validation absolute errors; it is not a formal probabilistic confidence interval.
- Weather, disease pressure, management decisions, and other annual drivers are absent.
- Changing vineyard area creates a scenario, but it does not establish a causal effect.
- The persistence forecast remains visible in the evaluation so that the ML model is not presented without a meaningful reference.